# M01Z – Migrating from Chat Completions to Responses API

This appendix is a **quick reference** for
developers who have existing code that uses the
older **Chat Completions** API.

You will see:

1. A simple **before/after** chat example
2. A **mapping table** between the two APIs
3. A **function calling** before/after example
4. A short **migration checklist**

For multi-turn conversations and advanced usage,
see **Module 4 (Instructions & Conversation
Chaining)**.

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

# Load .env
load_dotenv(dotenv_path=Path("..") / ".env")

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-5-mini"

print("✅ Ready!")

## Simple Chat Example – BEFORE (Chat Completions)

This is the **old** pattern using
`client.chat.completions.create`.

You do **not** need to run this code in this
course. It is here so you can see what you are
migrating *from*.

In [ ]:
# This is for reference only.
# It may not run in this course environment.

from openai import OpenAI

old_client = OpenAI()

old_response = old_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant.",
        },
        {
            "role": "user",
            "content": "What is the capital of France?",
        },
    ],
)

old_text = old_response.choices[0].message.content
print(old_text)

## Simple Chat Example – AFTER (Responses API)

Here is the **new** pattern using
`client.responses.create`.

Key changes:

- Use `instructions` for system-level behavior
- Use `input` for the actual user request
- Read `response.output_text` instead of
  `response.choices[0].message.content`

In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions="You are a helpful assistant.",
    input="What is the capital of France?",
)

print(response.output_text)

## Mapping: Chat Completions → Responses API

<div style="text-align: left; display: inline-block;">

| Concept                | Chat Completions                                         | Responses API                                    |
|-----------------------|------------------------------------------|------------------------------------------------|
| System prompt         | `{"role": "system"}` in `messages`   | `instructions="..."`                             |
| User message          | `{"role": "user"}` in `messages`     | `input="..."` or structured `input=[...]`        |
| Output text           | `choices[0].message.content`             | `output_text` or items in `response.output`      |
| Function / tool defs  | `tools=[{"type":"function","function":{...}}]` | `tools=[{"type":"function","name":...,"description":...,"parameters":...}]` |
| Multi-turn context    | Full `messages` history                  | `previous_response_id` + new `input`             |

</div>

For multi-turn conversations and context
management, see **Module 4**.

## Function Calling – BEFORE (Chat Completions)

Here is a simple function calling example using
the old Chat Completions API.

Again, this is for **reference only** so you can
see what is changing.

In [ ]:
# This is for reference only.
# It may not run in this course environment.

from openai import OpenAI

old_client = OpenAI()

old_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string"},
                },
                "required": ["city"],
            },
        },
    }
]

old_response = old_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": "What is the weather in Paris?",
        }
    ],
    tools=old_tools,
)

old_call = old_response.choices[0].message.tool_calls[0]
print("Function name:", old_call.function.name)
print("Arguments:", old_call.function.arguments)

## Function Calling – AFTER (Responses API)

Key differences in the **Responses API**:

- Tools use a **flattened structure** with
  `name`, `description`, `parameters` at the
  top level (not nested under a `function` key).
- The model returns `function_call` items inside
  `response.output`.
- You iterate over `response.output` and look for
  items where `item.type == "function_call"`.

If the model doesn't decide to call a tool,
`response.output` will contain only message text.

In a real app, you would execute your function
using `item.arguments`, then send the result back
to the model in a follow-up call (covered in
Module 6).

In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string"},
            },
            "required": ["city"],
        },
    }
]

response = client.responses.create(
    model=MODEL,
    input="What is the weather in Paris?",
    tools=tools,
)

for item in response.output:
    if item.type == "function_call":
        print("Function name:", item.name)
        print("Arguments:", item.arguments)

## Migration Checklist

When migrating from Chat Completions to the
Responses API:

1. Replace:
   - `client.chat.completions.create`  
   with:
   - `client.responses.create`
2. Move your **system** behavior into the
   `instructions="..."` parameter.
3. Use `input=...` for the **current user
   request** instead of building a `messages`
   list for single-turn calls.
4. For function calling:
   - Use a **flattened** structure: `name`,
     `description`, `parameters` at the top level
     (not nested under a `function` key)
   - Iterate over `response.output` and look for
     items with `item.type == "function_call"`.
5. For multi-turn conversations, do **not**
   rebuild long `messages` lists. Instead, use
   `previous_response_id` and new `input` as
   shown in **Module 4**.

Use this notebook as a quick reference whenever
you convert older Chat Completions code to the
modern Responses API.